## Libraries

In [139]:
import os
import pandas as pd

# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.linear_model import LogisticRegression
# from xgboost import XGBClassifier
# from imblearn.over_sampling import SMOTE
# from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc

## Load Dataset

In [140]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

Path to dataset files: /Users/madihamalik/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3


In [141]:
# Use correct file path and filename
filepath = os.path.join(path, "accepted_2007_to_2018Q4.csv.gz")

df = pd.read_csv(filepath, low_memory=False)
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## Filter out relevent columns for our research question

In [142]:
relevant_columns = [
    # Loan Characteristics
    "loan_amnt",
    "term",
    "int_rate",
    "grade",
    "purpose",
    # Borrower's Financials & History
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "home_ownership",
    "emp_length",
    "fico_range_low",
    "fico_range_high",
    "issue_d",
    "earliest_cr_line",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "verification_status",
    "application_type",
    "addr_state",
    # Target Variable
    "loan_status",
]
df = df[relevant_columns]
df.head()

,loan_amnt,term,int_rate,grade,purpose,annual_inc,dti,delinq_2yrs,inq_last_6mths,home_ownership,...,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,verification_status,application_type,addr_state,loan_status
0,3600.0,36 months,13.99,C,debt_consolidation,55000.0,5.91,0.0,1.0,MORTGAGE,...,Aug-2003,7.0,0.0,2765.0,29.7,13.0,Not Verified,Individual,PA,Fully Paid
1,24700.0,36 months,11.99,C,small_business,65000.0,16.06,1.0,4.0,MORTGAGE,...,Dec-1999,22.0,0.0,21470.0,19.2,38.0,Not Verified,Individual,SD,Fully Paid
2,20000.0,60 months,10.78,B,home_improvement,63000.0,10.78,0.0,0.0,MORTGAGE,...,Aug-2000,6.0,0.0,7869.0,56.2,18.0,Not Verified,Joint App,IL,Fully Paid
3,35000.0,60 months,14.85,C,debt_consolidation,110000.0,17.06,0.0,0.0,MORTGAGE,...,Sep-2008,13.0,0.0,7802.0,11.6,17.0,Source Verified,Individual,NJ,Current
4,10400.0,60 months,22.45,F,major_purchase,104433.0,25.37,1.0,3.0,MORTGAGE,...,Jun-1998,12.0,0.0,21929.0,64.5,35.0,Source Verified,Individual,PA,Fully Paid


### remove leading and trailing whitespace

In [143]:
df = df.apply(lambda col: col.str.strip() if col.dtypes == "object" else col)

## Target Variables

In [144]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

## Create binary target from loan_status

In [145]:
# Define which loan statuses mean the borrower defaulted (bad outcomes)
bad_statuses = ["Charged Off", "Default"]
df["is_default"] = df["loan_status"].apply(lambda x: 1 if x in bad_statuses else 0)

df.sample(5)

,loan_amnt,term,int_rate,grade,purpose,annual_inc,dti,delinq_2yrs,inq_last_6mths,home_ownership,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,verification_status,application_type,addr_state,loan_status,is_default
159136,19800.0,36 months,7.89,A,credit_card,109500.0,17.52,0.0,0.0,MORTGAGE,...,14.0,0.0,17718.0,31.9,29.0,Not Verified,Individual,IN,Fully Paid,0
2207735,10000.0,36 months,17.99,D,debt_consolidation,50000.0,21.72,1.0,1.0,MORTGAGE,...,14.0,0.0,6940.0,36.7,21.0,Source Verified,Individual,TX,Charged Off,1
1047434,10000.0,36 months,19.99,E,debt_consolidation,53000.0,35.71,0.0,1.0,MORTGAGE,...,16.0,0.0,16487.0,86.3,31.0,Source Verified,Individual,GA,Fully Paid,0
359405,20000.0,36 months,9.17,B,credit_card,85000.0,38.42,0.0,2.0,RENT,...,20.0,0.0,7813.0,19.5,45.0,Source Verified,Individual,CA,Fully Paid,0
1547730,2000.0,36 months,11.05,B,credit_card,19224.0,32.83,0.0,1.0,RENT,...,8.0,0.0,2790.0,16.2,15.0,Not Verified,Individual,PA,Current,0


## Create binary target from verification_status

In [146]:
# Define which statuses count as verified
verified_statuses = ["Verified", "Source Verified"]

# Create a new column 'is_verified': 1 if verified, 0 if not verified
df["is_verified"] = df["verification_status"].apply(
    lambda x: 1 if x in verified_statuses else 0
)
# Drop the Original Column:
df = df.drop("verification_status", axis=1)

df.sample(10)

,loan_amnt,term,int_rate,grade,purpose,annual_inc,dti,delinq_2yrs,inq_last_6mths,home_ownership,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type,addr_state,loan_status,is_default,is_verified
1982605,20000.0,36 months,14.49,C,credit_card,51000.0,28.28,0.0,0.0,OWN,...,10.0,0.0,25417.0,82.0,21.0,Individual,WA,Charged Off,1,0
846710,5400.0,36 months,8.46,A,debt_consolidation,58000.0,2.03,2.0,0.0,MORTGAGE,...,5.0,1.0,2421.0,47.5,11.0,Individual,GA,Current,0,1
1311463,15000.0,36 months,14.16,C,credit_card,75000.0,25.47,0.0,5.0,MORTGAGE,...,8.0,0.0,11127.0,80.1,18.0,Individual,IL,Fully Paid,0,1
1269669,18000.0,60 months,18.99,E,debt_consolidation,86400.0,26.75,0.0,0.0,MORTGAGE,...,18.0,0.0,20960.0,77.1,39.0,Individual,MO,Fully Paid,0,1
397742,6000.0,36 months,12.99,C,debt_consolidation,50000.0,14.67,1.0,0.0,MORTGAGE,...,8.0,0.0,9248.0,34.4,21.0,Individual,NY,Fully Paid,0,1
362820,5000.0,36 months,12.69,C,debt_consolidation,44000.0,4.15,0.0,1.0,MORTGAGE,...,8.0,0.0,3964.0,39.6,17.0,Individual,MI,Fully Paid,0,0
937250,6000.0,36 months,14.99,C,debt_consolidation,41695.0,18.71,0.0,0.0,RENT,...,7.0,0.0,14984.0,72.4,10.0,Individual,OK,Fully Paid,0,1
1819253,18000.0,36 months,11.55,B,debt_consolidation,70000.0,24.63,0.0,1.0,MORTGAGE,...,12.0,0.0,21874.0,72.7,32.0,Individual,FL,Fully Paid,0,0
145292,18000.0,60 months,16.99,D,debt_consolidation,60000.0,33.00,1.0,0.0,RENT,...,27.0,0.0,21112.0,77.0,43.0,Individual,CT,Fully Paid,0,1
1064798,15000.0,36 months,8.39,B,home_improvement,92000.0,22.55,0.0,0.0,OWN,...,14.0,0.0,13007.0,31.6,36.0,Individual,TX,Fully Paid,0,1


##  Filter for loans with known outcomes only

In [147]:
# Filter for loans with known outcomes only
resolved_statuses = ["Fully Paid", "Charged Off", "Default"]
df = df[df["loan_status"].isin(resolved_statuses)].copy()

## Drop columns that are no longer needed

In [148]:
# Drop columns that are no longer needed (safe to skip if already gone)
df = df.drop("loan_status", axis=1, errors="ignore")
df = df.drop("verification_status", axis=1, errors="ignore")

df.sample(6)

,loan_amnt,term,int_rate,grade,purpose,annual_inc,dti,delinq_2yrs,inq_last_6mths,home_ownership,...,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,application_type,addr_state,is_default,is_verified
1957653,22000.0,36 months,5.32,A,debt_consolidation,102000.0,3.91,0.0,0.0,MORTGAGE,...,Apr-2003,8.0,0.0,8649.0,29.0,26.0,Individual,TX,0,1
2122827,19200.0,60 months,13.59,C,major_purchase,40000.0,2.58,0.0,0.0,MORTGAGE,...,Feb-2005,4.0,0.0,8635.0,32.0,7.0,Individual,TN,0,1
654008,30000.0,36 months,22.39,E,credit_card,112000.0,27.34,0.0,0.0,MORTGAGE,...,Feb-2003,17.0,0.0,20237.0,64.4,25.0,Individual,OK,1,1
1154159,20000.0,60 months,15.61,D,credit_card,104760.0,18.78,0.0,2.0,RENT,...,Nov-2001,13.0,0.0,19943.0,53.8,41.0,Individual,IL,0,1
961804,11000.0,36 months,13.49,C,credit_card,78000.0,9.25,0.0,1.0,MORTGAGE,...,Jan-2005,9.0,0.0,9035.0,38.3,22.0,Individual,MN,0,0
1107086,8400.0,36 months,6.99,A,debt_consolidation,110000.0,7.38,1.0,1.0,MORTGAGE,...,Aug-1992,6.0,0.0,7751.0,43.1,24.0,Individual,MA,0,0


In [149]:
# Step 1: Keep only rows with valid categories
if "home_ownership" in df.columns:
    df = df[df["home_ownership"].isin(["MORTGAGE", "RENT", "OWN"])]

    # Step 2: One-hot encode and force 0/1 integers
    home_dummies = pd.get_dummies(
        df["home_ownership"], prefix="home", drop_first=False
    ).astype(int)

    # Step 3: Drop original column and add the new dummy columns
    df = df.drop("home_ownership", axis=1)
    df = pd.concat([df, home_dummies], axis=1)

# Step 4: Find and convert any existing dummy columns that are still boolean
dummy_cols = [col for col in df.columns if col.startswith("home_")]
df[dummy_cols] = df[dummy_cols].astype(int)

# Step 5: Drop any weird ones if they still exist
df = df.drop(
    columns=[
        col
        for col in [
            "home_NONE",
            "home_OTHER",
            "home_ownership_NONE",
            "home_ownership_OTHER",
        ]
        if col in df.columns
    ]
)
df.sample(6)

,loan_amnt,term,int_rate,grade,purpose,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,...,revol_bal,revol_util,total_acc,application_type,addr_state,is_default,is_verified,home_MORTGAGE,home_OWN,home_RENT
1276189,8000.0,36 months,11.67,B,debt_consolidation,40000.0,17.43,0.0,3.0,10+ years,...,6587.0,32.9,23.0,Individual,TN,0,0,1,0,0
961958,29125.0,60 months,12.74,C,debt_consolidation,132500.0,23.23,1.0,0.0,1 year,...,41395.0,44.4,39.0,Individual,FL,1,1,1,0,0
1715408,16000.0,36 months,17.99,D,debt_consolidation,63000.0,13.52,0.0,0.0,10+ years,...,17842.0,40.4,51.0,Individual,MS,0,1,1,0,0
1930645,35000.0,60 months,23.33,F,home_improvement,118000.0,14.08,0.0,3.0,2 years,...,18882.0,47.4,21.0,Individual,IL,0,1,1,0,0
6699,17000.0,36 months,8.49,B,credit_card,92500.0,17.23,2.0,1.0,10+ years,...,11391.0,55.0,14.0,Individual,MN,1,1,0,1,0
1051995,9000.0,36 months,6.97,A,debt_consolidation,45000.0,21.36,2.0,0.0,6 years,...,8902.0,62.3,23.0,Individual,AZ,0,0,0,0,1


## application_type
Values: Mostly "Individual" or "Joint App".
Since this column has only two categories, we can use binary encoding (0 and 1):

In [150]:
# Map "Individual" to 0 and "Joint App" to 1
df["application_type"] = df["application_type"].map({"Individual": 0, "Joint App": 1})
df.sample(6)

,loan_amnt,term,int_rate,grade,purpose,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,...,revol_bal,revol_util,total_acc,application_type,addr_state,is_default,is_verified,home_MORTGAGE,home_OWN,home_RENT
93331,8000.0,36 months,6.24,A,credit_card,72000.0,15.68,0.0,0.0,8 years,...,10968.0,70.8,14.0,0,NY,0,1,0,0,1
2176471,10000.0,36 months,8.24,B,credit_card,60000.0,6.30,2.0,0.0,10+ years,...,12058.0,44.5,12.0,0,NJ,0,0,1,0,0
296150,7500.0,36 months,6.92,A,credit_card,75000.0,22.96,0.0,1.0,10+ years,...,15619.0,48.8,28.0,0,IN,0,0,1,0,0
662805,4800.0,36 months,9.16,B,moving,90000.0,30.37,1.0,2.0,6 years,...,9156.0,83.0,26.0,0,CA,1,1,1,0,0
1033643,25000.0,60 months,22.45,E,debt_consolidation,142000.0,22.66,1.0,3.0,6 years,...,29179.0,68.2,24.0,0,CO,0,1,0,1,0
724502,24000.0,36 months,5.32,A,debt_consolidation,75000.0,19.22,0.0,0.0,8 years,...,25726.0,39.3,25.0,0,OH,0,0,1,0,0


### One-Hot Encode Top Categories in purpose

Keep the top 5 most frequent categories and group the rest as "other".

In [151]:
# Step 1: Get top 5 most frequent purposes
top_purposes = df["purpose"].value_counts().nlargest(5).index.tolist()

# Step 2: Replace rare ones with "other"
df["purpose"] = df["purpose"].apply(lambda x: x if x in top_purposes else "other")

# Step 3: One-hot encode with clear column names
purpose_dummies = pd.get_dummies(df["purpose"], prefix="purpose").astype(int)

# Example of how column names will look:
# purpose_debt_consolidation, purpose_small_business, purpose_home_improvement, etc.

# Step 4: Drop original and concat new columns
df = df.drop("purpose", axis=1)
df = pd.concat([df, purpose_dummies], axis=1)

df.sample(6)

,loan_amnt,term,int_rate,grade,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,fico_range_low,...,is_default,is_verified,home_MORTGAGE,home_OWN,home_RENT,purpose_credit_card,purpose_debt_consolidation,purpose_home_improvement,purpose_major_purchase,purpose_other
2070225,15000.0,36 months,14.08,C,500000.0,3.94,5.0,2.0,1 year,700.0,...,0,1,0,0,1,0,0,0,0,1
1275481,3150.0,36 months,10.99,B,63000.0,25.60,0.0,2.0,7 years,720.0,...,0,1,1,0,0,0,0,0,1,0
1074911,14000.0,60 months,15.31,C,101000.0,14.10,0.0,1.0,10+ years,705.0,...,1,1,0,0,1,0,0,0,1,0
387064,21000.0,36 months,13.66,C,130000.0,17.24,0.0,0.0,2 years,715.0,...,0,1,1,0,0,0,1,0,0,0
277357,25000.0,60 months,7.89,A,86000.0,20.15,1.0,0.0,10+ years,715.0,...,0,1,0,1,0,0,0,1,0,0
519079,5000.0,36 months,10.91,B,95000.0,9.21,0.0,3.0,10+ years,725.0,...,0,1,0,0,1,0,0,0,0,1


## Encode State and Prepare Data
One-hot encode the addr_state column, remove missing values, and convert dummy variables to integer format.

In [152]:
# One-hot encode addr_state
df_encoded = pd.get_dummies(df, columns=["addr_state"], drop_first=True)

# Drop rows with any NaN values
df_encoded = df_encoded.dropna()

# Find just the dummy columns that start with 'addr_state_'
addr_cols = [col for col in df_encoded.columns if col.startswith("addr_state_")]

# Convert only those to int
df_encoded[addr_cols] = df_encoded[addr_cols].astype(int)


df_encoded.sample(10)

,loan_amnt,term,int_rate,grade,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,fico_range_low,...,addr_state_SD,addr_state_TN,addr_state_TX,addr_state_UT,addr_state_VA,addr_state_VT,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY
306110,20000.0,36 months,7.89,A,128000.0,22.39,1.0,1.0,10+ years,675.0,...,0,0,0,0,0,0,0,0,0,0
714711,35000.0,36 months,11.47,B,250000.0,5.03,0.0,1.0,2 years,690.0,...,0,0,1,0,0,0,0,0,0,0
2178587,27825.0,36 months,17.99,D,120000.0,12.60,0.0,0.0,10+ years,665.0,...,0,0,0,0,0,0,0,0,0,0
1668796,20000.0,36 months,11.44,B,55000.0,34.91,0.0,0.0,10+ years,720.0,...,0,0,0,0,0,0,0,0,0,0
167836,16300.0,60 months,12.69,C,68000.0,19.62,0.0,0.0,2 years,720.0,...,0,0,0,1,0,0,0,0,0,0
40222,3000.0,36 months,11.22,B,40000.0,19.89,0.0,0.0,10+ years,665.0,...,0,0,1,0,0,0,0,0,0,0
2121359,15000.0,36 months,19.03,D,119600.0,19.38,0.0,0.0,10+ years,670.0,...,0,0,0,0,0,0,0,0,0,0
1247718,12000.0,36 months,10.99,B,72450.0,16.43,0.0,1.0,10+ years,760.0,...,0,0,0,0,0,0,1,0,0,0
568692,22000.0,36 months,30.99,G,50000.0,3.53,0.0,1.0,1 year,660.0,...,0,0,0,0,0,0,0,0,0,0
341324,13000.0,60 months,9.99,B,50000.0,23.64,0.0,0.0,3 years,710.0,...,0,0,0,0,0,0,0,0,0,0


### 🎯 Encode `grade` Feature
The `grade` column represents loan quality (A–G). map it to integers if you believe higher grades imply better creditworthiness.


In [153]:
# Safely one-hot encode "grade" with 0/1 values (if it exists)
if "grade" in df_encoded.columns:
    # One-hot encode and force int (0/1)
    grade_dummies = pd.get_dummies(
        df_encoded["grade"], prefix="grade", drop_first=False
    ).astype(int)

    # Drop the original "grade" column
    df_encoded = df_encoded.drop("grade", axis=1)

    # Concatenate the dummy columns back to df_encoded (not df)
    df_encoded = pd.concat([df_encoded, grade_dummies], axis=1)


# Display sample
df_encoded.sample(6)


,loan_amnt,term,int_rate,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,fico_range_low,fico_range_high,...,addr_state_WI,addr_state_WV,addr_state_WY,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G
1101784,10000.0,36 months,7.91,24000.0,30.75,0.0,0.0,1 year,680.0,684.0,...,0,0,0,1,0,0,0,0,0,0
366974,2400.0,36 months,8.18,15400.0,6.63,0.0,0.0,7 years,680.0,684.0,...,0,0,0,0,1,0,0,0,0,0
1306128,35000.0,36 months,11.99,145000.0,17.21,0.0,2.0,10+ years,725.0,729.0,...,0,0,0,0,1,0,0,0,0,0
976290,14000.0,36 months,17.99,43000.0,29.31,0.0,1.0,1 year,665.0,669.0,...,0,0,0,0,0,0,1,0,0,0
1970172,3500.0,36 months,17.99,95000.0,21.79,1.0,1.0,3 years,660.0,664.0,...,0,0,0,0,0,0,1,0,0,0
2093577,18400.0,36 months,16.02,45656.0,36.38,0.0,0.0,10+ years,680.0,684.0,...,0,0,0,0,0,1,0,0,0,0


### Clean term column ("36 months", "60 months")

In [154]:
# Extract numeric part and convert to integer
# Clean the 'term' column: from "36 months" -> 36 (int)
df_encoded["term"] = df_encoded["term"].astype(str).str.extract(r"(\d+)").astype(int)

df_encoded.sample(5)

,loan_amnt,term,int_rate,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,fico_range_low,fico_range_high,...,addr_state_WI,addr_state_WV,addr_state_WY,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G
1341720,18000.0,60,10.99,70000.0,6.79,1.0,0.0,9 years,725.0,729.0,...,0,0,0,0,1,0,0,0,0,0
2129876,15000.0,60,13.59,125000.0,22.08,0.0,0.0,10+ years,660.0,664.0,...,0,0,0,0,0,1,0,0,0,0
325506,7000.0,36,12.29,35000.0,20.88,1.0,1.0,4 years,680.0,684.0,...,0,0,0,0,0,1,0,0,0,0
1009142,12000.0,36,12.99,50000.0,24.58,0.0,0.0,3 years,680.0,684.0,...,0,0,0,0,0,1,0,0,0,0
91894,28250.0,36,5.32,62800.0,22.56,0.0,0.0,5 years,750.0,754.0,...,0,0,0,1,0,0,0,0,0,0


### Clean emp_length column ("10+ years", "< 1 year", "n/a")

In [155]:
def clean_emp_length(val):
    val = str(val).lower()  # convert everything to lowercase string
    if "< 1" in val:
        return 0
    elif "10+" in val:
        return 10
    elif "n/a" in val:
        return -1
    else:
        num = "".join([c for c in val if c.isdigit()])
        return int(num) if num else -1


# Apply the function
df_encoded["emp_length"] = df["emp_length"].apply(clean_emp_length).astype(int)

df_encoded.head(6)

,loan_amnt,term,int_rate,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,fico_range_low,fico_range_high,...,addr_state_WI,addr_state_WV,addr_state_WY,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G
0,3600.0,36,13.99,55000.0,5.91,0.0,1.0,10,675.0,679.0,...,0,0,0,0,0,1,0,0,0,0
1,24700.0,36,11.99,65000.0,16.06,1.0,4.0,10,715.0,719.0,...,0,0,0,0,0,1,0,0,0,0
2,20000.0,60,10.78,63000.0,10.78,0.0,0.0,10,695.0,699.0,...,0,0,0,0,1,0,0,0,0,0
4,10400.0,60,22.45,104433.0,25.37,1.0,3.0,3,695.0,699.0,...,0,0,0,0,0,0,0,0,1,0
5,11950.0,36,13.44,34000.0,10.20,0.0,0.0,4,690.0,694.0,...,0,0,0,0,0,1,0,0,0,0
6,20000.0,36,9.17,180000.0,14.67,0.0,0.0,10,680.0,684.0,...,0,0,0,0,1,0,0,0,0,0


In [156]:
missing_counts = df_encoded.isnull().sum()
print(missing_counts[missing_counts > 0])

Series([], dtype: int64)


In [157]:
print(df_encoded["is_default"].value_counts(normalize=True))

is_default
0    0.804663
1    0.195337
Name: proportion, dtype: float64


In [158]:
df_encoded.select_dtypes(include=["object"]).head()

,issue_d,earliest_cr_line
0,Dec-2015,Aug-2003
1,Dec-2015,Dec-1999
2,Dec-2015,Aug-2000
4,Dec-2015,Jun-1998
5,Dec-2015,Oct-1987


**Convert dates to date-time type**

In [159]:
# 4. Engineer credit history length
# Check if the original date columns exist
if "issue_d" in df_encoded.columns and "earliest_cr_line" in df_encoded.columns:
    print("Engineering 'credit_history_length'...")
    # Ensure columns are datetime before performing operations
    df_encoded["issue_d"] = pd.to_datetime(df["issue_d"])
    df_encoded["earliest_cr_line"] = pd.to_datetime(df_encoded["earliest_cr_line"])
    df_encoded["credit_history_length"] = (
        (df_encoded["issue_d"] - df_encoded["earliest_cr_line"]).dt.days / 30
    ).round(0)
    # Drop original date columns now that we're done with them
    df_encoded = df_encoded.drop(
        columns=["issue_d", "earliest_cr_line"], errors="ignore"
    )

df_encoded.head()

Engineering 'credit_history_length'...


/var/folders/sh/fp8tjdc13dz_ghxdvpqwnhhm0000gn/T/ipykernel_59914/1354202584.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_encoded["issue_d"] = pd.to_datetime(df["issue_d"])
/var/folders/sh/fp8tjdc13dz_ghxdvpqwnhhm0000gn/T/ipykernel_59914/1354202584.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_encoded["earliest_cr_line"] = pd.to_datetime(df_encoded["earliest_cr_line"])


,loan_amnt,term,int_rate,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,fico_range_low,fico_range_high,...,addr_state_WV,addr_state_WY,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,credit_history_length
0,3600.0,36,13.99,55000.0,5.91,0.0,1.0,10,675.0,679.0,...,0,0,0,0,1,0,0,0,0,150.0
1,24700.0,36,11.99,65000.0,16.06,1.0,4.0,10,715.0,719.0,...,0,0,0,0,1,0,0,0,0,195.0
2,20000.0,60,10.78,63000.0,10.78,0.0,0.0,10,695.0,699.0,...,0,0,0,1,0,0,0,0,0,187.0
4,10400.0,60,22.45,104433.0,25.37,1.0,3.0,3,695.0,699.0,...,0,0,0,0,0,0,0,1,0,213.0
5,11950.0,36,13.44,34000.0,10.20,0.0,0.0,4,690.0,694.0,...,0,0,0,0,1,0,0,0,0,343.0


**Strip leading/tailing white spaces**

In [160]:
df["emp_length"].value_counts()

emp_length
10+ years    442075
2 years      121712
< 1 year     108008
3 years      107558
1 year        88468
5 years       84112
4 years       80525
6 years       62706
8 years       60689
7 years       59606
9 years       50921
Name: count, dtype: int64

In [161]:
# Convert 'issue_d' and 'earliest_cr_line' to datetime format
df_encoded["issue_d"] = pd.to_datetime(df_encoded["issue_d"], format="%b-%Y")
df_encoded["earliest_cr_line"] = pd.to_datetime(
    df_encoded["earliest_cr_line"], format="%b-%Y"
)

# Create a new column: length of credit history in months
df_encoded["credit_history_months"] = (
    df_encoded["issue_d"] - df_encoded["earliest_cr_line"]
).dt.days // 30


KeyError: 'issue_d'

In [ ]:
df_encoded["credit_history_months"].describe()


count    1.344872e+06
mean     1.974722e+02
std      9.139075e+01
min      3.600000e+01
25%      1.360000e+02
50%      1.790000e+02
75%      2.430000e+02
max      1.013000e+03
Name: credit_history_months, dtype: float64

In [165]:
# Drop any "grade_" columns that still have NaNs
grade_cols_with_nans = [
    col
    for col in df_encoded.columns
    if col.startswith("grade_") and df_encoded[col].isnull().any()
]

# Drop those columns from the DataFrame
df_encoded = df_encoded.drop(columns=grade_cols_with_nans)

df_encoded.columns


Index(['loan_amnt', 'term', 'int_rate', 'annual_inc', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'emp_length', 'fico_range_low', 'fico_range_high',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'application_type', 'is_default', 'is_verified', 'home_MORTGAGE',
       'home_OWN', 'home_RENT', 'purpose_credit_card',
       'purpose_debt_consolidation', 'purpose_home_improvement',
       'purpose_major_purchase', 'purpose_other', 'addr_state_AL',
       'addr_state_AR', 'addr_state_AZ', 'addr_state_CA', 'addr_state_CO',
       'addr_state_CT', 'addr_state_DC', 'addr_state_DE', 'addr_state_FL',
       'addr_state_GA', 'addr_state_HI', 'addr_state_IA', 'addr_state_ID',
       'addr_state_IL', 'addr_state_IN', 'addr_state_KS', 'addr_state_KY',
       'addr_state_LA', 'addr_state_MA', 'addr_state_MD', 'addr_state_ME',
       'addr_state_MI', 'addr_state_MN', 'addr_state_MO', 'addr_state_MS',
       'addr_state_MT', 'addr_state_NC', 'addr_state_ND', 'addr_stat

In [171]:
correlation_matrix = df_encoded.corr()


In [174]:
# Remove rows with invalid (negative) debt-to-income (DTI) values
df_encoded = df_encoded[df_encoded["dti"] >= 0]


In [175]:
df_encoded.describe()


,loan_amnt,term,int_rate,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,fico_range_low,fico_range_high,...,addr_state_WV,addr_state_WY,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,credit_history_length
count,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,...,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06,1.265557e+06
mean,1.460310e+04,4.191729e+01,1.323245e+01,7.788790e+04,1.813101e+01,3.213336e-01,6.568270e-01,5.966208e+00,6.961156e+02,7.001157e+02,...,3.530461e-03,2.210884e-03,1.756286e-01,2.921283e-01,2.833543e-01,1.483916e-01,6.966972e-02,2.398075e-02,6.846788e-03,1.942929e+02
std,8.745474e+03,1.034412e+01,4.769235e+00,7.102512e+04,9.569814e+00,8.833241e-01,9.393550e-01,3.691052e+00,3.165674e+01,3.165733e+01,...,5.931273e-02,4.696805e-02,3.805040e-01,4.547412e-01,4.506271e-01,3.554879e-01,2.545897e-01,1.529892e-01,8.246159e-02,8.766764e+01
min,5.000000e+02,3.600000e+01,5.310000e+00,3.300000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.250000e+02,6.290000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.600000e+01
25%,8.000000e+03,3.600000e+01,9.750000e+00,4.800000e+04,1.176000e+01,0.000000e+00,0.000000e+00,2.000000e+00,6.700000e+02,6.740000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.350000e+02
50%,1.210000e+04,3.600000e+01,1.274000e+01,6.500000e+04,1.752000e+01,0.000000e+00,0.000000e+00,6.000000e+00,6.900000e+02,6.940000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.780000e+02
75%,2.000000e+04,3.600000e+01,1.599000e+01,9.250000e+04,2.391000e+01,0.000000e+00,1.000000e+00,1.000000e+01,7.100000e+02,7.140000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.390000e+02
max,4.000000e+04,6.000000e+01,3.099000e+01,1.099920e+07,9.990000e+02,3.900000e+01,8.000000e+00,1.000000e+01,8.450000e+02,8.500000e+02,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.014000e+03


In [178]:
df_encoded["is_default"].value_counts()


is_default
0    1018347
1     247210
Name: count, dtype: int64

# Save your cleaned DataFrame to the 1_datasets/processed_data

In [179]:
df_encoded.to_csv("lendingclub_cleaned_encoded_with_address.csv", index=False)


### 📁 Cleaned dataset available here: [Download lending_club_cleaned.csv](https://drive.google.com/drive/folders/1qNO8Zt4Hla22DKdx6A3FxfRT9koHN-0A)
